# LAB 4 — Using MCP in LangChain

This notebook walks through all 8 steps of the lab:
1. Setup & installation
2. Connect to MCP server
3. Load MCP tools into LangChain
4. Create agent with MCP tools
5. Access MCP resources
6. Build a complete MCP-enabled agent
7. Compare MCP vs direct API integration
8. Practical example — Document Analysis Agent

## Step 1 — Setup & Installation

In [ ]:
%pip install -q langchain langchain-openai langchain-mcp-adapters mcp python-dotenv

In [ ]:
import asyncio   # needed to run async MCP calls from a regular Jupyter cell
import os
from pathlib import Path

from dotenv import load_dotenv

# Load OPENAI_API_KEY from .env into the process environment
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")
assert api_key, "OPENAI_API_KEY not found — check your .env file"
print(f"API key loaded: {api_key[:8]}...")

# Resolve the documents folder relative to this notebook's location
DOCS_DIR = str(Path(".") / "documents")
print(f"Documents directory: {DOCS_DIR}")

## Step 2 — Connect to MCP Server

We use `MultiServerMCPClient` from `langchain-mcp-adapters`.  
The filesystem MCP server is launched via `npx` (requires Node.js).

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# SERVER_CONFIG maps a friendly server name to its launch command.
# "transport": "stdio" means the client communicates with the server
# over stdin/stdout of the spawned npx process (no HTTP port needed).
# "-y" tells npx to auto-install the package if it isn't cached yet.
SERVER_CONFIG = {
    "filesystem": {
        "command": "npx",
        "args": ["-y", "@modelcontextprotocol/server-filesystem", DOCS_DIR],
        "transport": "stdio",
    }
}

print("MCP server config ready.")
print(f"  Server : @modelcontextprotocol/server-filesystem")
print(f"  Root   : {DOCS_DIR}")

## Step 3 — Load MCP Tools into LangChain

In [ ]:
async def load_tools(client: MultiServerMCPClient):
    # get_tools() queries every connected MCP server and returns
    # LangChain BaseTool objects — no manual wrapping needed.
    tools = await client.get_tools()
    print(f"Loaded {len(tools)} MCP tool(s):")
    for t in tools:
        print(f"  • {t.name}: {t.description[:90]}")
    return tools

async def _connect_and_load():
    global _client, mcp_tools
    # Manually enter the async context so the client stays open
    # across multiple cells (instead of closing after a single `async with` block).
    _client = MultiServerMCPClient(SERVER_CONFIG)
    await _client.__aenter__()
    mcp_tools = await load_tools(_client)

# asyncio.run() is used because Jupyter cells are not themselves async functions.
asyncio.run(_connect_and_load())

## Step 4 — Create Agent with MCP Tools

In [ ]:
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# temperature=0 makes responses deterministic — better for document Q&A
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=api_key)

# The prompt has three parts:
#   1. system  — tells the agent its role and where docs live
#   2. human   — the user's question at runtime ({input})
#   3. agent_scratchpad — placeholder where LangChain injects tool call history
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a helpful document-analysis assistant. "
        "You have access to a filesystem MCP server. "
        "Always use the available tools to read real file content before answering. "
        f"Documents are stored in: {DOCS_DIR}"
    ),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),  # required by create_tool_calling_agent
])

# create_tool_calling_agent uses the LLM's native tool-calling API (no ReAct text parsing)
agent = create_tool_calling_agent(llm, mcp_tools, prompt)

# AgentExecutor is the runtime loop: LLM → tool call → observation → LLM → ...
executor = AgentExecutor(agent=agent, tools=mcp_tools, verbose=True)

print("Agent created with", len(mcp_tools), "MCP tool(s).")

## Step 5 — Access MCP Resources

The `@modelcontextprotocol/server-filesystem` server exposes directory listings
and file contents as **resources** (read-only context) in addition to **tools** (actions).  
With the stdio transport the adapter surfaces these through the `list_directory` and
`read_file` tools, so the agent accesses them via tool calls rather than a separate
resource URI. The cell below demonstrates reading a resource directly before the
agent is involved.

In [ ]:
async def preview_resource(filename: str):
    # Find the read_file tool from the loaded MCP tools list
    read_tool = next(t for t in mcp_tools if "read" in t.name.lower())
    # ainvoke calls the MCP tool asynchronously — equivalent to the agent doing it
    result = await read_tool.ainvoke({"path": str(Path(DOCS_DIR) / filename)})
    print(f"--- Resource preview: {filename} ---")
    print(str(result)[:600])  # truncate so the output stays readable
    print("...")

# Preview one document directly via MCP tool (bypassing the agent)
asyncio.run(preview_resource("mcp_overview.txt"))

## Step 6 — Complete MCP-Enabled Agent Demo

Four queries that exercise different MCP tool calls.

In [ ]:
# These four queries cover the full range of MCP tool usage:
#   - list_directory: discover what files exist
#   - read_file: fetch a single document
#   - multiple read_file calls: cross-document comparison
QUERIES = [
    "List all files available in the documents directory.",
    "Read the file about MCP and give me a one-paragraph summary.",
    "What are the top 3 AI trends mentioned in the ai_trends_2025 document?",
    "Compare all three documents and identify the common theme.",
]

async def run_queries():
    for i, query in enumerate(QUERIES, 1):
        print(f"\n{'='*60}")
        print(f"Query {i}: {query}")
        print("="*60)
        # ainvoke runs the full agent loop asynchronously
        result = await executor.ainvoke({"input": query})
        print(f"\nAnswer:\n{result['output']}")

asyncio.run(run_queries())

## Step 7 — Compare MCP vs Direct API Integration

In [ ]:
# ── Direct API approach ──────────────────────────────────────────────────────
# Instead of letting the agent call MCP tools, we manually read the file
# in Python and pass its content straight to the LLM.
# This is simpler but requires custom I/O code for every data source.

file_path = Path(DOCS_DIR) / "mcp_overview.txt"
content = file_path.read_text(encoding="utf-8")  # manual file read — no MCP

# Build a plain chat messages list (no tools, no agent loop)
messages = [
    {"role": "system", "content": "Answer based on the document provided."},
    {"role": "user", "content": f"Document:\n{content}\n\nQuestion: What problem does MCP solve?"},
]

response = llm.invoke(messages)  # single LLM call, no tool loop
print("Direct API answer:")
print(response.content)

In [ ]:
comparison = """
COMPARISON: MCP vs Direct API
==============================
MCP Integration:
  + Standardised protocol — reusable across LangChain, CrewAI, Claude Desktop, etc.
  + Agent discovers tools dynamically — no manual plumbing per data source
  + Scales easily: add a new MCP server = add a new block to SERVER_CONFIG
  - Requires an MCP server process (npx child process here)
  - Extra abstraction layer adds startup latency and a small debugging surface

Direct API Integration:
  + Simple, fast, full control over every API call
  + No extra processes or dependencies
  - Custom code per data source (file I/O, HTTP, auth, parsing)
  - Not reusable across frameworks without rewriting

Rule of thumb:
  1 data source, 1 framework  →  Direct API
  Multiple sources or frameworks  →  MCP
"""
print(comparison)

## Step 8 — Practical Example: Document Analysis Agent

A focused set of real-world document-analysis queries using the MCP-powered agent.

In [ ]:
# Step 8: real-world queries that combine multiple MCP tool calls in one agent run
PRACTICAL_QUERIES = [
    # Requires reading all 3 files to search for "RAG"
    "Search all documents for any mention of 'RAG' and explain what it means in context.",
    # Requires reading all 3 files then synthesising a combined summary
    "Generate a combined executive summary of all documents in three bullet points.",
]

async def run_practical():
    for i, query in enumerate(PRACTICAL_QUERIES, 1):
        print(f"\n{'='*60}")
        print(f"Practical Query {i}: {query}")
        print("="*60)
        result = await executor.ainvoke({"input": query})
        print(f"\nAnswer:\n{result['output']}")

asyncio.run(run_practical())

In [ ]:
# Always close the MCP client when done.
# __aexit__ terminates the npx child process and frees the stdio pipes.
async def cleanup():
    await _client.__aexit__(None, None, None)
    print("MCP client disconnected.")

asyncio.run(cleanup())